In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math

# ==============================================================================
# 0. SymPy settings
# ==============================================================================
sp.init_printing(use_unicode=True)
z = sp.Symbol('z', complex=True)
q = sp.Symbol('q', complex=True)
n = sp.Symbol('n', integer=True)
I = sp.I
pi = sp.pi

print("=== LTI System Step Response Analysis — Problem 090601 ===")
print()

# ==============================================================================
# 1. SYSTEM COEFFICIENTS & TRANSFER FUNCTION H(z)
# ==============================================================================
alpha_1 = sp.Rational(-9, 10)
alpha_2 = sp.Rational(81, 100)
beta_0 = sp.Integer(1)

A_den = sp.expand(1 + alpha_1*q + alpha_2*q**2)
H_z = sp.factor(beta_0 / A_den).subs(q, z**(-1))
H_q = sp.factor(beta_0 / A_den)

display(Math(r"H(z)=\frac{1}{1-0.9z^{-1}+0.81z^{-2}}"))
display(sp.Eq(sp.Symbol("H(z)"), H_z))

# ==============================================================================
# 2. POLES
# ==============================================================================
p1 = sp.Rational(9, 10) * sp.exp(I*pi/3)
p2 = sp.Rational(9, 10) * sp.exp(-I*pi/3)
H_pole_form = sp.factor(1 / ((1-p1*q)*(1-p2*q))).subs(q, z**(-1))
H_pole_form_q = sp.factor(1 / ((1-p1*q)*(1-p2*q)))

# ==============================================================================
# 3. UNIT-STEP INPUT X(z)
# ==============================================================================
X_z = sp.factor(1/(1-z**(-1)))
X_q = sp.factor(1/(1-q))
display(Math(r"x[n]=u[n]"))
display(sp.Eq(sp.Symbol("X(z)"), X_z))

# ==============================================================================
# 4. ZERO-STATE RESPONSE Y_zs(z) & PFE
# ==============================================================================
Y_zs = sp.factor(H_pole_form * X_z)
Y_zs_q = sp.factor(H_pole_form_q * X_q)
display(sp.Eq(sp.Symbol(r"Y_{zs}(z)"), Y_zs))

A, B, C = sp.symbols('A B C')
Y_zs_trial_q = A/(1-p1*q) + B/(1-p2*q) + C/(1-q)
numerator_zs = sp.expand(sp.together(Y_zs_trial_q - Y_zs_q).as_numer_denom()[0])
poly_zs = sp.Poly(numerator_zs, q)
equations_zs = [sp.Eq(coeff, 0) for coeff in poly_zs.all_coeffs()]
solution_zs = sp.solve(equations_zs, [A, B, C], dict=True, simplify=True)[0]

A_zs = sp.simplify(solution_zs[A])
B_zs = sp.simplify(solution_zs[B])
C_zs = sp.simplify(solution_zs[C])

print("\nZero-State PFE Numerical Coefficients:")
display(sp.N(A_zs, 10))
display(sp.N(B_zs, 10))
display(sp.N(C_zs, 10))

Y_zs_pfe = sp.simplify((A/(1-p1*q) + B/(1-p2*q) + C/(1-q)).subs(solution_zs)).subs(q, z**(-1))
display(Y_zs_pfe)

mag_A_zs = sp.simplify(sp.Abs(A_zs))
phase_A_zs = sp.simplify(sp.arg(A_zs))
mag_p = sp.simplify(sp.Abs(p1))
phase_p = sp.simplify(sp.arg(p1))

y_zs = sp.simplify(C_zs + 2*mag_A_zs*mag_p**n * sp.cos(n*phase_p + phase_A_zs))

# ==============================================================================
# 5. ZERO-INPUT RESPONSE Y_zi(z) & PFE
# ==============================================================================
y_minus_1 = sp.Integer(1)
y_minus_2 = sp.Integer(1)
D_q = sp.simplify(-alpha_1*y_minus_1 - alpha_2*y_minus_2*q - alpha_2*y_minus_2)
D_z = D_q.subs(q, z**(-1))
Y_zi = sp.factor(D_q / A_den).subs(q, z**(-1))
Y_zi_q = sp.factor(D_q / A_den)

A_zi, B_zi = sp.symbols('A_zi B_zi')
Y_zi_trial_q = A_zi/(1-p1*q) + B_zi/(1-p2*q)
numerator_zi = sp.expand(sp.together(Y_zi_trial_q - Y_zi_q).as_numer_denom()[0])
poly_zi = sp.Poly(numerator_zi, q)
equations_zi = [sp.Eq(coeff, 0) for coeff in poly_zi.all_coeffs()]
solution_zi = sp.solve(equations_zi, [A_zi, B_zi], dict=True, simplify=True)[0]

A_zi_value = sp.simplify(solution_zi[A_zi])
B_zi_value = sp.simplify(solution_zi[B_zi])

print("\nZero-Input PFE Numerical Coefficients:")
display(sp.N(A_zi_value, 10))
display(sp.N(B_zi_value, 10))

Y_zi_pfe = sp.simplify((A_zi/(1-p1*q) + B_zi/(1-p2*q)).subs(solution_zi)).subs(q, z**(-1))
display(Y_zi_pfe)

mag_A_zi = sp.simplify(sp.Abs(A_zi_value))
phase_A_zi = sp.simplify(sp.arg(A_zi_value))
y_zi = sp.simplify(2*mag_A_zi*mag_p**n * sp.cos(n*phase_p + phase_A_zi))

# ==============================================================================
# 6. TOTAL RESPONSE (Zero State + Zero Input)
# ==============================================================================
y_total = sp.simplify(y_zs + y_zi)
Y_total_z = sp.factor(Y_zs + Y_zi)

display(sp.Eq(sp.Symbol(r"Y_{total}(z)"), Y_total_z))
display(Math(r"y[n] = y_{zs}[n] + y_{zi}[n]"))
display(y_total)

# ==============================================================================
# 7. NUMERICAL EVALUATION & PLOTTING
# ==============================================================================
def evaluate_symbolic(expr, n_values):
    return np.array([float(sp.re(sp.N(expr.subs(n, int(k))))) for k in n_values])

out = widgets.Output()

def plot_system_responses(N=20):
    with out:
        out.clear_output(wait=True)
        n_vec = np.arange(0, N)
        zs_vals = evaluate_symbolic(y_zs, n_vec)
        zi_vals = evaluate_symbolic(y_zi, n_vec)
        total_vals = evaluate_symbolic(y_total, n_vec)

        fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

        axes[0].stem(n_vec, zs_vals, basefmt=" ")
        axes[0].set_title(r"Zero-State Response $y_{zs}[n]$", fontsize=11, fontweight='bold')
        axes[0].set_ylabel(r"$y_{zs}[n]$")
        axes[0].grid(True)

        axes[1].stem(n_vec, zi_vals, basefmt=" ", linefmt='r-', markerfmt='ro')
        axes[1].set_title(r"Zero-Input Response $y_{zi}[n]$ ($y[-1]=y[-2]=1$)", fontsize=11, fontweight='bold')
        axes[1].set_ylabel(r"$y_{zi}[n]$")
        axes[1].grid(True)

        axes[2].stem(n_vec, total_vals, basefmt=" ", linefmt='g-', markerfmt='go')
        axes[2].set_title(r"Total Response $y[n]=y_{zs}[n]+y_{zi}[n]$", fontsize=11, fontweight='bold')
        axes[2].set_xlabel(r"$n$")
        axes[2].set_ylabel(r"$y[n]$")
        axes[2].grid(True)

        plt.tight_layout()
        plt.show()

plot_system_responses()
display(out)